<a href="https://colab.research.google.com/github/CodeCrusaders26/privacy-dns-filter/blob/feature%2Fml-model/notebooks/processed_dataset_v2_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Task 1: Inspect the Original Dataset

We will inspect `/content/combined_domains.csv` to look at:
- Dataset shape, column names, and data types.
- The first 10 rows to understand the representation of domains, sources, and labels.
- Missing values and duplicate domain entries.

In [1]:
import pandas as pd

# Load the dataset
file_path = '/content/combined_domains.csv'
df = pd.read_csv(file_path)

# Display shape and basic info
print("=== Dataset Shape ===")
print(df.shape)

print("\n=== Data Types and Info ===")
print(df.info())

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== First 10 Rows ===")
display(df.head(10))

# Identify unique counts for domain and source/label columns
print("\n=== Column Value Counts & Statistics ===")
for col in df.columns:
    unique_count = df[col].nunique()
    print(f"Column '{col}' has {unique_count} unique values.")
    if unique_count < 20:
        print(df[col].value_counts(dropna=False))

# Check for duplicate domains
if 'domain' in df.columns:
    duplicate_count = df['domain'].duplicated().sum()
    print(f"\nDuplicate domains in dataset: {duplicate_count}")
elif 'Domain' in df.columns:
    duplicate_count = df['Domain'].duplicated().sum()
    print(f"\nDuplicate domains in dataset: {duplicate_count}")
else:
    # Fallback to check duplicates for the first column if no named domain column is detected
    duplicate_count = df.iloc[:, 0].duplicated().sum()
    print(f"\nDuplicate entries in the first column: {duplicate_count}")

=== Dataset Shape ===
(19950, 4)

=== Data Types and Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19950 entries, 0 to 19949
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   domain           19950 non-null  object
 1   label            19950 non-null  int64 
 2   source           19950 non-null  object
 3   collection_date  19950 non-null  object
dtypes: int64(1), object(3)
memory usage: 623.6+ KB
None

=== Missing Values ===
domain             0
label              0
source             0
collection_date    0
dtype: int64

=== First 10 Rows ===


,domain,label,source,collection_date
0,findmykids.org,0,Tranco,2026-11-11
1,labs.google,0,Tranco,2026-11-11
2,integrating.smbc-flexibler.com,1,PhishTank,2026-11-11
3,unl.edu,0,Tranco,2026-11-11
4,ggi.dietondemand.com,1,URLhaus,2026-11-11
5,bnf.fr,0,Tranco,2026-11-11
6,qualys.com,0,Tranco,2026-11-11
7,t.yesware.com,1,PhishTank,2026-11-11
8,tagesschau.de,0,Tranco,2026-11-11
9,flashscore.ua,0,Tranco,2026-11-11



=== Column Value Counts & Statistics ===
Column 'domain' has 19950 unique values.
Column 'label' has 2 unique values.
label
1    10000
0     9950
Name: count, dtype: int64
Column 'source' has 4 unique values.
source
Tranco               9950
PhishTank            9016
URLhaus               976
PhishTank+URLhaus       8
Name: count, dtype: int64
Column 'collection_date' has 1 unique values.
collection_date
2026-11-11    19950
Name: count, dtype: int64

Duplicate domains in dataset: 0


### Task 2.1: Source-Label Validation & Normalization Impact Check

We will check for domain normalization issues and cross-reference sources against labels to ensure perfect logical consistency before altering any data.

In [2]:
import pandas as pd

# Ensure we are using the loaded df
# 1. Cross-tabulation of source vs label
print("=== Source vs Label Cross-tabulation ===")
cross_tab = pd.crosstab(df['source'], df['label'])
display(cross_tab)

# 2. Inspect the 8 records with source 'PhishTank+URLhaus'
print("\n=== Records from PhishTank+URLhaus ===")
duall_source_df = df[df['source'] == 'PhishTank+URLhaus']
display(duall_source_df)

# 3. Test impact of normalization on domain uniqueness
# We will test lowercasing, stripping whitespace, and removing trailing dots (if any).
normalized_series = df['domain'].astype(str).str.lower().str.strip().str.rstrip('.')
original_unique_count = df['domain'].nunique()
normalized_unique_count = normalized_series.nunique()

print(f"\n=== Normalization Impact Analysis ===")
print(f"Original unique domains count: {original_unique_count}")
print(f"Normalized unique domains count: {normalized_unique_count}")

if original_unique_count != normalized_unique_count:
    print("\nWARNING: Normalization will merge some domain strings! Let's find them:")
    # Identify which domains are causing conflicts or becoming duplicates
    temp_df = df.copy()
    temp_df['normalized_domain'] = normalized_series
    dup_normalized = temp_df[temp_df['normalized_domain'].duplicated(keep=False)].sort_values(by='normalized_domain')
    display(dup_normalized.head(20))
else:
    print("\nExcellent: Domain normalization will not create any duplicate domain records.")

=== Source vs Label Cross-tabulation ===


label,0,1
source,,
PhishTank,0,9016
PhishTank+URLhaus,0,8
Tranco,9950,0
URLhaus,0,976



=== Records from PhishTank+URLhaus ===


,domain,label,source,collection_date
1947,j-0-09g-9bh-h-ggf-1324992141.cos.ap-bangkok.my...,1,PhishTank+URLhaus,2026-11-11
2965,drive.google.com,1,PhishTank+URLhaus,2026-11-11
3396,firebasestorage.googleapis.com,1,PhishTank+URLhaus,2026-11-11
7107,f3i5-0g49bgn-3h95-1324992141.cos.ap-jakarta.my...,1,PhishTank+URLhaus,2026-11-11
7373,www.dropbox.com,1,PhishTank+URLhaus,2026-11-11
9107,docs.google.com,1,PhishTank+URLhaus,2026-11-11
12945,bj5y6-0f-9h4-9fgg4-1324992141.cos.ap-bangkok.m...,1,PhishTank+URLhaus,2026-11-11
18238,linktr.ee,1,PhishTank+URLhaus,2026-11-11



=== Normalization Impact Analysis ===
Original unique domains count: 19950
Normalized unique domains count: 19950

Excellent: Domain normalization will not create any duplicate domain records.
